# `03` Output Control, HITL & Audit — The 70% Threshold
### Worksheet Step 3 · Layers 5–7 · ⏱ ~10 min
> **Setup:** standard library only — run cells top to bottom (`Shift`+`Enter`). See `00_Overview` for environment setup.

> **Worksheet — the 62% test:** *Your AI returns a result at 62% confidence.
> It's below the 70% threshold. Map the exact steps the faculty reviewer takes.*

This notebook implements the heart of the talk: a **confidence gate** that lets
confident answers through and routes uncertain ones to a human, plus a
**tamper-evident audit log**.

In [1]:
THRESHOLD = 0.70   # the dial from the talk — tune per task sensitivity

def combine_confidence(model_conf: float, retrieval_score: float) -> float:
    """Blend the model's self-confidence with how well RAG grounded it."""
    return round(0.6 * model_conf + 0.4 * min(1.0, retrieval_score * 2), 2)

def gate(confidence: float):
    if confidence >= THRESHOLD:
        return "AUTO_DELIVER"
    return "HUMAN_REVIEW"

for c in [0.92, 0.71, 0.62, 0.40]:
    print(f"confidence {c} -> {gate(c)}")

confidence 0.92 -> AUTO_DELIVER
confidence 0.71 -> AUTO_DELIVER
confidence 0.62 -> HUMAN_REVIEW
confidence 0.4 -> HUMAN_REVIEW


## Tamper-evident audit log (Layer 7)

Every decision is appended to a hash-chained log: each entry includes the hash
of the previous one, so any edit to history breaks the chain. This is the
"Exemplary" governance criterion in the rubric.

In [2]:
import hashlib, json, time

class AuditLog:
    def __init__(self):
        self.entries = []
    def append(self, **event):
        prev = self.entries[-1]["hash"] if self.entries else "GENESIS"
        payload = {"ts": round(time.time(), 3), "prev": prev, **event}
        payload["hash"] = hashlib.sha256(
            (prev + json.dumps(event, sort_keys=True)).encode()).hexdigest()[:16]
        self.entries.append(payload)
        return payload["hash"]
    def verify(self):
        prev = "GENESIS"
        for e in self.entries:
            ev = {k: v for k, v in e.items() if k not in ("ts", "prev", "hash")}
            h = hashlib.sha256((prev + json.dumps(ev, sort_keys=True)).encode()).hexdigest()[:16]
            if h != e["hash"] or e["prev"] != prev:
                return False
            prev = e["hash"]
        return True

audit = AuditLog()
audit.append(action="query", user="faculty01", conf=0.62, decision="HUMAN_REVIEW")
audit.append(action="review", user="faculty01", outcome="approved_with_edits")
print("chain valid?", audit.verify())
audit.entries[0]["conf"] = 0.95          # tamper!
print("chain valid after tamper?", audit.verify())

chain valid? True
chain valid after tamper? False


## The HITL workflow — the 62% test in code

This is exactly what the worksheet asks you to map. Fill in `review_steps`
with the *actual* steps your reviewer takes.

In [3]:
def hitl_review(item, reviewer="faculty"):
    # The exact steps the worksheet asks you to define:
    review_steps = [
        "1. See the answer, its confidence, and the cited sources side by side",
        "2. Check each claim against the citation (Layer-4 grounding)",
        "3. Edit / correct, or reject and request regeneration",
        "4. Record the decision + rationale to the audit log",
    ]
    for s in review_steps:
        print("   ", s)
    return {"decision": "approved_with_edits", "reviewer": reviewer}

def secure_respond(query, model_conf, retrieval_score, citations, audit):
    conf = combine_confidence(model_conf, retrieval_score)
    decision = gate(conf)
    print(f"Query: {query!r}\nBlended confidence: {conf} -> {decision}")
    audit.append(action="respond", conf=conf, decision=decision,
                 citations=citations)
    if decision == "HUMAN_REVIEW":
        print("Routing to human. Reviewer steps:")
        out = hitl_review(query)
        audit.append(action="human_decision", **out)
        return {"status": "delivered_after_review", "confidence": conf}
    return {"status": "auto_delivered", "confidence": conf}

# the 62% test:
secure_respond("Re-grade this borderline essay", model_conf=0.6,
               retrieval_score=0.31, citations=["rubric_essay"], audit=audit)

Query: 'Re-grade this borderline essay'
Blended confidence: 0.61 -> HUMAN_REVIEW
Routing to human. Reviewer steps:
    1. See the answer, its confidence, and the cited sources side by side
    2. Check each claim against the citation (Layer-4 grounding)
    3. Edit / correct, or reject and request regeneration
    4. Record the decision + rationale to the audit log


{'status': 'delivered_after_review', 'confidence': 0.61}

### ✏️ Your turn
1. Rewrite `review_steps` to match **your** scenario's reviewer.
2. **Guard the reviewer:** add a rule that auto-delivers trivially safe queries
   so humans aren't buried (the fatigue-mitigation point on the worksheet).
3. Move `THRESHOLD` up/down — which tasks deserve a stricter dial?